In [23]:
!pip install python-docx

In [24]:
from docx import Document

doc = Document('/Users/michael/Documents/ml.docx')

for i, paragraph in enumerate(doc.paragraphs, 1):
    text = paragraph.text.strip()

    if not text:
        continue
    words = text.split()
    first_few_words = " ".join(words[:3])

    print(f"Paragraph {i}")
    print(f"First words: {first_few_words}")
    print("-"*60)
    

Paragraph 1
First words: Машинное обучение —
------------------------------------------------------------
Paragraph 2
First words: Существует три основных
------------------------------------------------------------
Paragraph 3
First words: Глубокое обучение, являющееся
------------------------------------------------------------
Paragraph 4
First words: Применение машинного обучения
------------------------------------------------------------
Paragraph 5
First words: Несмотря на впечатляющие
------------------------------------------------------------


In [25]:
from typing import List

def fixsized_chunk(text:str, chunk_size: int=500, overlap: int=50) -> List[str]:
    if not text or chunk_size <= 0:
        return []

    chunks = []
    start = 0
    text_len = len(text)


    while start < text_len:
        end = min(start+chunk_size, text_len)

        if end < text_len and text[end] != ' ' and text[end-1] != ' ':
            last_space = text.rfind(' ', start, end)
            if last_space > start:
                end = last_space

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap if end < text_len else text_len
        if start < text_len and text[start] != ' ' and text[start-1] != ' ':
            next_space = text.find(' ', start, end)
            if next_space < end:
                start = next_space

    return chunks

paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
text = "\n\n".join(paragraphs)
chunks = fixsized_chunk(text, chunk_size=500, overlap=50)
for i, c in enumerate(chunks, 1):
    print(f"Chunk {i}: {c}")

Chunk 1: Машинное обучение — это раздел искусственного интеллекта, в котором компьютерные системы учатся выполнять задачи, не будучи явно запрограммированными для каждой конкретной ситуации. Вместо того чтобы следовать жёсткому набору инструкций, написанных человеком, алгоритмы анализируют большие объёмы данных, выявляют в них закономерности и на основе этих закономерностей строят прогнозы или принимают решения. Этот подход кардинально отличается от классического программирования, где разработчик должен
Chunk 2: программирования, где разработчик должен заранее предусмотреть все возможные сценарии.

Существует три основных типа машинного обучения: обучение с учителем, обучение без учителя и обучение с подкреплением. При обучении с учителем алгоритму подаются данные с готовыми ответами — например, тысячи фотографий кошек и собак, каждая из которых подписана. Обучение без учителя работает с неразмеченными данными и пытается самостоятельно найти скрытую структуру, например сгруппировать кл

In [26]:
import sys
!{sys.executable} -m pip install  llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface chromadb sentence-transformers

In [27]:
import uuid

In [28]:
from llama_index.core import Document
from llama_index.core.node_parser import TokenTextSplitter

doc = Document(text = text, metadata={"source":"Machine Learning 101", "author":"Qwen3.7+"})

splitter = TokenTextSplitter(chunk_size=200, chunk_overlap=20)
nodes = splitter.get_nodes_from_documents([doc])

for node in nodes:
    node.metadata["chunk_id"] = uuid.uuid4().hex[:8]
    node.excluded_embed_metadata_keys = []

for node in nodes:
    print(f"-----Chunk {node.metadata["chunk_id"]} | All metadata: {node.metadata}-----")
    print(node.get_content())

-----Chunk 4feedb4d | All metadata: {'source': 'Machine Learning 101', 'author': 'Qwen3.7+', 'chunk_id': '4feedb4d'}-----
Машинное обучение — это раздел искусственного интеллекта, в котором компьютерные системы учатся выполнять задачи, не будучи явно запрограммированными для каждой конкретной ситуации. Вместо того чтобы следовать жёсткому набору инструкций, написанных человеком, алгоритмы анализируют большие объёмы данных, выявляют в них закономерности и на основе этих закономерностей строят прогнозы или принимают решения. Этот подход
-----Chunk aaa0ac42 | All metadata: {'source': 'Machine Learning 101', 'author': 'Qwen3.7+', 'chunk_id': 'aaa0ac42'}-----
прогнозы или принимают решения. Этот подход кардинально отличается от классического программирования, где разработчик должен заранее предусмотреть все возможные сценарии.

Существует три основных типа машинного обучения: обучение с учителем, обучение без учителя и обучение с подкреплением. При обучении с учителем алгоритму подаются дан

In [29]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter

chroma_client = chromadb.HttpClient(host="localhost", port=8000)
chroma_collection = chroma_client.get_or_create_collection("ml_docs_collection")

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True
)

print("Data loaded!")

retriever = index.as_retriever(similarity_top_k=3)

query = "На чем основано глубокое обучение?"

print(f"-------Search query {query}--------")
retrieved_nodes = retriever.retrieve(query)
for i, node_with_score in enumerate(retrieved_nodes, 1):
    node = node_with_score.node
    score = node_with_score.score

    print(f"Similarity: {score:.4f}")
    print(f"Metadata: {node.metadata}")
    print(f"-Answer: {node.get_content()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/8 [00:00<?, ?it/s]

Data loaded!
-------Search query На чем основано глубокое обучение?--------
Similarity: 0.4529
Metadata: {'source': 'Machine Learning 101', 'author': 'Qwen3.7+', 'chunk_id': '8591d14b'}
-Answer: понимать человеческую речь или генерировать реалистичные изображения. Именно глубокое обучение стоит за такими прорывами, как голосовые ассистенты, автопилоты и современные языковые модели.

Применение машинного обучения сегодня охватывает практически все сферы жизни. В медицине алгоритмы помогают обнаруживать опухоли на снимках МРТ с точностью, сопоставимой с опытными врачами. В
Similarity: 0.4483
Metadata: {'source': 'Machine Learning 101', 'author': 'Qwen3.7+', 'chunk_id': 'bb375782'}
-Answer: понимать человеческую речь или генерировать реалистичные изображения. Именно глубокое обучение стоит за такими прорывами, как голосовые ассистенты, автопилоты и современные языковые модели.

Применение машинного обучения сегодня охватывает практически все сферы жизни. В медицине алгоритмы помогают обна

In [30]:
!{sys.executable} -m pip install llama-index-llms-ollama

In [31]:
from llama_index.llms.ollama import Ollama

llm = Ollama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    request_timeout=120.0
)
print("LLM is running!")

LLM is running!


In [32]:
filters = MetadataFilters(
    filters=[
        MetadataFilter(key="chunk_type", value="full_paragraph", operator="==")
    ]
)

query_engine_filtered = index.as_query_engine(
    similarity_top_k=3,
    llm=llm,
    filters=filters
)

query = "В чем проблемы машинного обучения?"
response_filtered = query_engine_filtered.query(query)

print(f"Filtered response for query: {query}")
print(response_filtered.response)
for node in response_filtered.source_nodes:
    print(f"Chunk type: {node.node.metadata["chunk_type"]} | Score: {node.score:.4f}")

Filtered response for query: В чем проблемы машинного обучения?
Empty Response
